# Khabar Segmentation - Fine-tuning CAMeL-BERT-CA on Google Colab

Pipeline complet pour entraîner CAMeL-BERT-CA sur la segmentation d'akhbars avec contexte réel.

**Améliorations par rapport à AraBERT:**
- CAMeL-BERT-CA: optimisé pour l'arabe classique
- Dataset avec contexte O réel (pas juste des akhbars isolés)
- 424 exemples avec alignement de qualité (score ≥ 85)

**GPU recommandé:** T4 ou L4 (gratuit)
**Durée estimée:** ~15-20 minutes

## 1. Setup & Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("[OK] Google Drive montée!")

## 2. Vérifier le GPU

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Cloner/Accéder au repo

In [ ]:
import os
import subprocess

# Chemin du repo sur Drive
repo_path = '/content/Khabar-segmentation'
drive_repo_path = '/content/drive/MyDrive/Khabar-segmentation'

if not os.path.exists(repo_path):
    print("[*] Copying repo from Drive...")
    if os.path.exists(drive_repo_path):
        subprocess.run(['cp', '-r', drive_repo_path, repo_path])
        print(f"[OK] Repo copied: {repo_path}")
    else:
        print(f"[!] Repo not found at {drive_repo_path}")
else:
    print(f"[OK] Repo already exists: {repo_path}")

os.chdir(repo_path)
print(f"[OK] Working directory: {os.getcwd()}")

## 4. Installer les dépendances

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pyarabic tqdm rapidfuzz

## 5. Vérifier le dataset

In [ ]:
import json
from pathlib import Path

dataset_path = Path('./data/processed/train_akhbars_with_context.jsonl')

if not dataset_path.exists():
    print(f"[!] Dataset not found: {dataset_path}")
    print("    Run prepare_dataset_with_context.py locally first")
else:
    # Charger et afficher quelques exemples
    examples = []
    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
                if len(examples) >= 3:
                    break
    
    print(f"[OK] Dataset loaded: {len(examples)} examples shown")
    for i, ex in enumerate(examples):
        print(f"\n[EXAMPLE {i+1}] Akhbar #{ex['akhbar_num']}")
        print(f"  Tokens: {len(ex['tokens'])} (length: {ex['length']})")
        print(f"  Labels: {ex['labels'][:20]}...")  # Premiers 20 labels
        label_counts = {}
        for label in ex['labels']:
            label_counts[label] = label_counts.get(label, 0) + 1
        print(f"  Label distribution: {label_counts}")

## 6. Fine-tuner CAMeL-BERT-CA

In [ ]:
import json
from pathlib import Path
from typing import Dict, List
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm


# Label encoding
LABEL2ID = {
    'O': 0,
    'B-KHABAR': 1,
    'I-KHABAR': 2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("[*] Configuration")
print("    Model: CAMeL-Lab/bert-base-arabic-camelbert-ca")
print("    Task: Token classification (BIO) for akhbar segmentation with context")
print("    Epochs: 15")
print("    Batch size: 8")
print("    Learning rate: 2e-5")
print()

In [ ]:
# Chemins
data_dir = Path('./data/processed')
dataset_path = data_dir / 'train_akhbars_with_context.jsonl'
output_dir = Path('./checkpoints/camelbert_akhbars_v2')

# Vérifier le dataset
if not dataset_path.exists():
    print(f"[!] Dataset not found: {dataset_path}")
else:
    print(f"[OK] Dataset found: {dataset_path}")
    output_dir.mkdir(parents=True, exist_ok=True)

print(f"[OK] Output directory: {output_dir}")

In [ ]:
def load_examples_from_jsonl(jsonl_path: str) -> List[Dict]:
    """Charger les exemples depuis un fichier JSONL."""
    examples = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"Loading {Path(jsonl_path).name}"):
            if line.strip():
                examples.append(json.loads(line))
    return examples


def tokenize_and_align_labels(
    examples: Dict,
    tokenizer,
    max_length: int = 512
) -> Dict:
    """Tokenizer et aligner les labels avec les tokens du tokenizer."""
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        padding='max_length',
        return_overflowing_tokens=False,
    )

    labels = []
    for i, label_seq in enumerate(examples['labels']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                if word_idx < len(label_seq):
                    label_ids.append(LABEL2ID.get(label_seq[word_idx], 0))
                else:
                    label_ids.append(-100)
            else:
                if word_idx < len(label_seq):
                    label = label_seq[word_idx]
                    if label.startswith('B-'):
                        label = 'I-' + label[2:]
                    label_ids.append(LABEL2ID.get(label, 0))
                else:
                    label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs


def compute_metrics(eval_pred) -> Dict[str, float]:
    """Calculer les metriques d'evaluation."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for prediction, label in zip(predictions, labels):
        for pred, lbl in zip(prediction, label):
            if lbl != -100:
                true_predictions.append(ID2LABEL[pred])
                true_labels.append(ID2LABEL[lbl])

    print("\n[METRICS]")
    print(classification_report(true_labels, true_predictions))

    return {
        'f1': f1_score(true_labels, true_predictions, average='weighted'),
        'precision': precision_score(true_labels, true_predictions, average='weighted', zero_division=0),
        'recall': recall_score(true_labels, true_predictions, average='weighted', zero_division=0),
    }

print("[OK] Functions defined")

In [ ]:
print("[*] Loading examples...")
examples = load_examples_from_jsonl(str(dataset_path))
print(f"[OK] Loaded {len(examples)} examples")

# Statistiques
lengths = [ex['length'] for ex in examples]
print(f"\n[DATASET STATISTICS]")
print(f"  Min length: {min(lengths)} tokens")
print(f"  Max length: {max(lengths)} tokens")
print(f"  Mean length: {sum(lengths) / len(lengths):.0f} tokens")

# Label distribution
label_counts = {'O': 0, 'B-KHABAR': 0, 'I-KHABAR': 0}
for ex in examples:
    for label in ex['labels']:
        label_counts[label] = label_counts.get(label, 0) + 1

total = sum(label_counts.values())
print(f"\n[LABEL DISTRIBUTION]")
for label, count in label_counts.items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

In [ ]:
print("[*] Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/bert-base-arabic-camelbert-ca')

model = AutoModelForTokenClassification.from_pretrained(
    'CAMeL-Lab/bert-base-arabic-camelbert-ca',
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("[OK] Model loaded")

In [ ]:
print("[*] Preparing dataset...")

# Convertir en Dataset HF
dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in examples],
    'labels': [ex['labels'] for ex in examples],
})

# Split 80/20 pour validation locale
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"  Train: {len(train_dataset)} examples")
print(f"  Val: {len(val_dataset)} examples")

# Tokenizer et aligner
print("[*] Tokenizing and aligning labels...")
train_dataset = train_dataset.map(
    lambda x: tokenize_and_align_labels(x, tokenizer, max_length=512),
    batched=True,
    remove_columns=['tokens', 'labels'],
    desc="Train"
)

val_dataset = val_dataset.map(
    lambda x: tokenize_and_align_labels(x, tokenizer, max_length=512),
    batched=True,
    remove_columns=['tokens', 'labels'],
    desc="Val"
)

print("[OK] Dataset prepared")

In [ ]:
# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir=str(output_dir),
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    save_strategy='steps',
    save_steps=50,
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=3,
    seed=42,
    report_to=[],
)

# Trainer
print("[*] Creating trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("[OK] Trainer ready")

In [ ]:
print("\n" + "="*70)
print("[*] Starting training...")
print(f"    Model: CAMeL-BERT-CA (Classical Arabic)")
print(f"    Dataset: 424 examples with real context (O labels)")
print(f"    Output: {output_dir}")
print("="*70 + "\n")

trainer.train()

print("\n" + "="*70)
print("[OK] Training complete!")
print("="*70)

In [ ]:
print("[*] Saving model...")
trainer.save_model(str(output_dir))

# Sauvegarder les labels
with open(output_dir / 'label2id.json', 'w') as f:
    json.dump(LABEL2ID, f)
with open(output_dir / 'id2label.json', 'w') as f:
    json.dump(ID2LABEL, f)

print(f"[OK] Model saved to {output_dir}")
print("[OK] Labels saved!")

## 7. Quick Test - Tester sur quelques akhbars annotés

In [ ]:
import json
from pathlib import Path
import torch
from transformers import pipeline
import numpy as np

print("[*] Loading model for testing...")
nlp = pipeline(
    'token-classification',
    model=str(output_dir),
    tokenizer='CAMeL-Lab/bert-base-arabic-camelbert-ca',
    device=0 if torch.cuda.is_available() else -1
)

print("[*] Loading test akhbars...")
json_path = Path('./data/processed/Kitab_Uqala_al_Majanin_annotated.json')

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Prendre les 5 premiers akhbars
test_akhbars = []
for akh in data['akhbar'][:5]:
    text = ' '.join([seg['text'] for seg in akh['content']['segments']])
    test_akhbars.append({
        'num': akh['num'],
        'text': text,
        'length': len(text.split())
    })

print(f"[OK] Loaded {len(test_akhbars)} akhbars\n")

stats = {'total': len(test_akhbars), 'recognized': 0, 'confidences': []}

for akh_info in test_akhbars:
    print(f"[AKHBAR {akh_info['num']}] ({akh_info['length']} words)")
    print(f"Text: {akh_info['text'][:150]}...\n")

    try:
        sentence = akh_info['text']
        if len(sentence) > 2000:
            sentence = sentence[:2000]

        predictions = nlp(sentence)

        khabar_tokens = []
        khabar_scores = []

        for pred in predictions:
            entity = pred.get('entity', 'O')
            score = pred.get('score', 0.0)
            word = pred.get('word', '')

            if 'KHABAR' in entity:
                khabar_tokens.append(word)
                khabar_scores.append(score)

        if khabar_tokens:
            avg_score = np.mean(khabar_scores) if khabar_scores else 0.0
            stats['recognized'] += 1
            stats['confidences'].append(avg_score)

            print(f"✓ Recognized: YES")
            print(f"  Confidence: {avg_score:.4f}")
            print(f"  Tokens recognized: {len(khabar_tokens)}")
        else:
            print(f"✗ Recognized: NO")

    except Exception as e:
        print(f"Error: {e}")

    print()

print("="*70)
print(f"[RESULTS]")
print(f"  Total tested: {stats['total']}")
print(f"  Recognized: {stats['recognized']}/{stats['total']}")
if stats['confidences']:
    print(f"  Average confidence: {np.mean(stats['confidences']):.4f}")
print()

## 8. Sauvegarder les résultats sur Google Drive

In [ ]:
import shutil
from pathlib import Path

# Sauvegarder le checkpoint du modèle
checkpoint_src = Path('checkpoints/camelbert_akhbars_v2')
checkpoint_dst = Path('/content/drive/MyDrive/Khabar-segmentation-results/checkpoints/camelbert_akhbars_v2')

if checkpoint_src.exists():
    checkpoint_dst.parent.mkdir(parents=True, exist_ok=True)
    if checkpoint_dst.exists():
        shutil.rmtree(checkpoint_dst)
    shutil.copytree(checkpoint_src, checkpoint_dst)
    print(f"[OK] Checkpoint saved to Drive: {checkpoint_dst}")
else:
    print(f"[!] Checkpoint not found: {checkpoint_src}")

print("\n[OK] Training complete and backed up!")

## 9. Résumé

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════╗
║         KHABAR SEGMENTATION - CAMeL-BERT-CA TRAINING            ║
╚═══════════════════════════════════════════════════════════════╝

[OK] Fine-tuning complete!

Model: checkpoints/camelbert_akhbars_v2
Dataset: 424 examples with real O context
Base: CAMeL-BERT-CA (Classical Arabic)
Epochs: 15

Key improvements over AraBERT:
  ✓ CAMeL-BERT-CA optimized for Classical Arabic
  ✓ Real O labels around akhbars (not isolated examples)
  ✓ 84.96 average alignment quality
  ✓ Better boundary detection

Next steps:
  1. Download checkpoint from Drive
  2. Run evaluate.py on OpenITI corpus
  3. Analyze extraction quality
  4. Fine-tune parameters if needed

Questions? Check CLAUDE.md
""")